In [ ]:
import pyodbc

# اتصال به SQL Server
conn = pyodbc.connect(
    # 'DRIVER={SQL Server};'
    # 'SERVER=MKZ-DSAS\\DSAS;'
    # 'DATABASE=DSAS;'
    # 'UID=datadriven;'
    # 'PWD=5Rdx@4Rfv1355'
    'DRIVER={SQL Server};'
    'SERVER=10WKS-PISHVA;'
    'DATABASE=PEGAH;'
    'UID=rezapishva;'
    'PWD=5rdx@4rfv1355'
)

cursor = conn.cursor()

# لیست AssetIDهایی که می‌خوای بررسی بشن
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# لیست برای ذخیره مقادیر Value
values = []

# اجرای کوئری برای هر AssetID و دریافت فقط Value
for asset_id in asset_ids:
    query = f"""
        SELECT TOP 1 [Value]
        FROM [PDA].[Periodic_Values]
        WHERE UnitID = 11 AND AssetID = {asset_id}
        ORDER BY DateTime DESC
    """
    cursor.execute(query)
    row = cursor.fetchone()
    if row:
        values.append(row.Value)
    else:
        values.append(None)  


value_8341, value_8342, value_8343, value_8344, value_8346, value_9286, value_9287 = values

# نمایش مقادیر
print("✅ مقادیر آخرین رکوردها برای UnitID=11:")
print(f"AssetID 8341 → Value: {value_8341}")
print(f"AssetID 8342 → Value: {value_8342}")
print(f"AssetID 8343 → Value: {value_8343}")
print(f"AssetID 8344 → Value: {value_8344}")
print(f"AssetID 8346 → Value: {value_8346}")
print(f"AssetID 9286 → Value: {value_9286}")
print(f"AssetID 9287 → Value: {value_9287}")

# بستن اتصال
cursor.close()
conn.close()




import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import joblib

# بارگذاری اجزای مدل
scaler = joblib.load('scaler.pkl')
dbscan = joblib.load('dbscan_model.pkl')
cluster_points = np.load('cluster_points.npy')

# تابع تشخیص ناهنجاری با آستانه وزن
def is_anomalous(input_dict, threshold=10.0):
    input_df = pd.DataFrame([input_dict])
    scaled_input = scaler.transform(input_df)
    label = dbscan.fit_predict(scaled_input)[0]

    if label != -1:
        return {'is_anomaly': False, 'anomaly_weight': 0.0}

    # محاسبه فاصله از نزدیک‌ترین خوشه
    distance = pairwise_distances(scaled_input, cluster_points).min()
    anomaly_weight = distance

    # بررسی آستانه
    is_anomaly = anomaly_weight > threshold
    return {'is_anomaly': is_anomaly, 'anomaly_weight': anomaly_weight}

# مثال استفاده
sample_input = {
    'AssetID_8341': value_8341,
    'AssetID_8342': value_8342,
    'AssetID_8343': value_8343,
    'AssetID_8344': value_8344,
    'AssetID_8346': value_8346,
    'AssetID_9286': value_9286,
    'AssetID_9287': value_9287
}

result = is_anomalous(sample_input)
print(result)



import mysql.connector
from datetime import datetime

# اتصال به دیتابیس MySQL
conn = mysql.connector.connect(
    host='127.0.0.1',
    port=3306,
    user='root',
    password='',  
    database='dsas'
)

cursor = conn.cursor()

# مقادیر ورودی
inputs = [value_8341,value_8342,value_8343,value_8344,value_8346,value_9286,value_9287]
anomaly_weight = result['anomaly_weight']  # مقدار score
results = "Normal" if anomaly_weight < 5 else "Abnormal"
model_name = "Anomaly detection for lube oil system"
unitID = 11
system = "dbscan clustering weighted by computing distance from clusters "
score = anomaly_weight
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
updated_at = created_at

print("1")

query = """
    INSERT INTO results_dsas_mhi_lube_oil_11 
    (inputs, results, model_name, unitID, system, score, created_at, updated_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""
print("2")

cursor.execute(query, (
    str(inputs),  # تبدیل لیست به رشته برای ذخیره در فیلد text یا varchar
    results,
    model_name,
    unitID,
    system,
    score,
    created_at,
    updated_at
))
print("3")
# ذخیره تغییرات
conn.commit()
print("4")
print("✅ داده با موفقیت ثبت شد.")

# بستن اتصال
cursor.close()
conn.close()
